# Stage 1 — Qwen3-32B value axis (Colab A100)

Three **separate** stages (do not combine into one long cell):

1. **Extract** — GPU, long-running, Drive resume via `activations_32b/<conv_id>.npz`
2. **Build axis** — CPU-ok, needs complete (or enough) extract caches
3. **Gate** — CPU-ok, needs `value_axis_32b.npy` + held-out activations

**Runtime:** A100 80GB for extract (bf16 ~64GB). Build/gate can run after disconnect without the big model.

**Checkpoints:** mount Drive early. Caches are **labeled-token-only** (pre/post discovery tokens, all 64 layers) — Drive-sized, not full-sequence. Re-run **extract only** after a disconnect — existing `.npz` files are skipped. Do **not** set `FORCE_EXTRACT=True` unless you intend a full redo.

## Two ICRL upload paths (run **one** upload cell, not both)

| Path | Upload cell | Source file | When to use |
|------|-------------|-------------|-------------|
| **Faithful (production)** | "Upload — Opus / OpenRouter" | `icrl_32b.json` | Final 32B axis |
| **Proxy smoke test** | "Upload — proxy ICRL" | `icrl_proxy.json` | Smoke only — not paper-faithful |

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need a GPU runtime (A100)'
print(torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
REPO = '/content/failure_prediction_research'
if not os.path.isdir(REPO):
    # Prefer your fork URL if different:
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}/stage1
!pip install -q -e .
!pip install -q openai python-dotenv

In [ ]:
# Checkpoint root on Google Drive (survives Colab disconnects)
# Each finished conversation -> activations_32b/<conv_id>.npz
# Format: labeled_v1 (pre/post tokens only, all layers) — typically tens of MB/conv, not ~1GB.
# Re-running extract SKIPS existing .npz files unless FORCE_EXTRACT=True.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/failure_prediction_research/stage1_32b')
ACT_DIR = DRIVE_ROOT / 'activations_32b'
AXIS_DIR = DRIVE_ROOT / 'artifacts'
ACT_DIR.mkdir(parents=True, exist_ok=True)
AXIS_DIR.mkdir(parents=True, exist_ok=True)

FORCE_EXTRACT = False  # True only if you want to redo ALL activations from scratch

n_done = len(list(ACT_DIR.glob('*.npz')))
print('Drive root:', DRIVE_ROOT)
print('Activations dir:', ACT_DIR)
print('Cached npz so far:', n_done)
print('FORCE_EXTRACT:', FORCE_EXTRACT)
if n_done:
    sizes = [p.stat().st_size for p in ACT_DIR.glob('*.npz')]
    print(f'size MB: min={min(sizes)/1e6:.1f} median={sorted(sizes)[len(sizes)//2]/1e6:.1f} '
          f'max={max(sizes)/1e6:.1f} total={sum(sizes)/1e9:.2f} GB')

## Choose ICRL source

Run **exactly one** of the next two cells:

1. **Opus / OpenRouter (faithful)** — if you generated `icrl_32b.json` on your laptop with Claude Opus 4.6.
2. **Proxy smoke** — if you only have the old `icrl_proxy.json` (Qwen-local generator) and want to test 32B extract + gate before OpenRouter credits are ready.

Skip the other upload cell entirely.

In [ ]:
# Upload — Opus / OpenRouter (faithful production path)
# Generate on laptop:
#   python -m stage1.icrl_gen.generate --n 300 --backend openrouter \
#     --target-model Qwen3-32B --min-paragraphs 3 --max-paragraphs 8 \
#     --output data/icrl_32b.json --resume
# Then upload the resulting icrl_32b.json here.
# Do NOT run the proxy upload cell below if you use this cell.

from pathlib import Path
from google.colab import files
from stage1.common.paths import data_file

ICRL = data_file('icrl_32b.json')
ICRL.parent.mkdir(parents=True, exist_ok=True)
if not ICRL.exists():
    print('Upload icrl_32b.json (Claude Opus 4.6 via OpenRouter)...')
    up = files.upload()
    src = Path(next(iter(up)))
    src.rename(ICRL)
print('ICRL path:', ICRL)
print('exists:', ICRL.exists())
if ICRL.exists():
    print('size bytes:', ICRL.stat().st_size)

In [ ]:
# Upload — proxy ICRL (smoke test only)
# Use old icrl_proxy.json (Qwen-local generator, ~67 convs from Downloads).
# Purpose: verify Qwen3-32B load + extract + gate wiring before faithful Opus data exists.
# Not paper-faithful — treat output as a smoke axis, not for Stage-2 science.
# Do NOT run the Opus upload cell above if you use this cell.

from pathlib import Path
from google.colab import files
from stage1.common.paths import data_file

ICRL = data_file('icrl_proxy.json')
ICRL.parent.mkdir(parents=True, exist_ok=True)
if not ICRL.exists():
    print('Upload icrl_proxy.json (proxy / Qwen-local dataset)...')
    up = files.upload()
    src = Path(next(iter(up)))
    src.rename(ICRL)
print('ICRL path:', ICRL)
print('exists:', ICRL.exists())
if ICRL.exists():
    print('size bytes:', ICRL.stat().st_size)

In [ ]:
# === STAGE 1/3: EXTRACT ONLY ===
# Writes ACT_DIR/<conv_id>.npz on Drive. Resume-safe (skips existing unless FORCE_EXTRACT).
# After disconnect: remount Drive, re-upload ICRL if needed, re-run THIS cell only.
# Quiet at start = loading Qwen3-32B; then "saved icrl_..." lines.
import subprocess, sys
from pathlib import Path

assert 'ACT_DIR' in dir(), 'Run the Google Drive checkpoint cell first'
assert 'ICRL' in dir() and Path(ICRL).exists(), 'Run an ICRL upload cell first'

cmd = [
    sys.executable, '-u', '-m', 'stage1.pipeline.run_gate',
    '--preset', 'qwen32b',
    '--stage', 'extract',
    '--icrl', str(ICRL),
    '--activations-dir', str(ACT_DIR),
]
if FORCE_EXTRACT:
    cmd.append('--force-extract')

print('STAGE: extract', flush=True)
print('ICRL:', ICRL, flush=True)
print('ACT_DIR:', ACT_DIR, flush=True)
print('cached npz before:', len(list(ACT_DIR.glob('*.npz'))), flush=True)
print('Quiet while loading Qwen3-32B is normal...', flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('extract exit code:', rc, flush=True)
print('cached npz after:', len(list(ACT_DIR.glob('*.npz'))), flush=True)

In [ ]:
# === STAGE 2/3: BUILD AXIS ONLY ===
# No GPU model load. Uses ACT_DIR/*.npz → value_axis_32b.npy
# Run after extract is complete (or after enough train-split caches exist).
import subprocess, sys, shutil
from pathlib import Path
from stage1.common.paths import data_file

assert 'ACT_DIR' in dir(), 'Run the Google Drive checkpoint cell first'
n = len(list(ACT_DIR.glob('*.npz')))
assert n > 0, f'No activations in {ACT_DIR} — run extract first'

cmd = [
    sys.executable, '-u', '-m', 'stage1.pipeline.run_gate',
    '--preset', 'qwen32b',
    '--stage', 'build',
    '--activations-dir', str(ACT_DIR),
]
print('STAGE: build', flush=True)
print('ACT_DIR:', ACT_DIR, 'npz:', n, flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('build exit code:', rc, flush=True)

src = data_file('value_axis_32b.npy')
if src.exists():
    dst = AXIS_DIR / 'value_axis_32b.npy'
    shutil.copy2(src, dst)
    print('copied to Drive:', dst, flush=True)
else:
    print('missing locally:', src, flush=True)

In [ ]:
# === STAGE 3/3: GATE ONLY ===
# Loads value_axis_32b.npy + held-out activations → AUROC + gate pass/fail.
# No GPU model load. Run after build.
import subprocess, sys, shutil
from pathlib import Path
from stage1.common.paths import data_file

assert 'ACT_DIR' in dir(), 'Run the Google Drive checkpoint cell first'
axis = data_file('value_axis_32b.npy')
assert axis.exists(), f'Missing {axis} — run build first'

cmd = [
    sys.executable, '-u', '-m', 'stage1.pipeline.run_gate',
    '--preset', 'qwen32b',
    '--stage', 'gate',
    '--activations-dir', str(ACT_DIR),
]
if 'ICRL' in dir() and Path(ICRL).exists():
    cmd.extend(['--icrl', str(ICRL)])

print('STAGE: gate', flush=True)
print('axis:', axis, flush=True)
print('ACT_DIR:', ACT_DIR, 'npz:', len(list(ACT_DIR.glob('*.npz'))), flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('gate exit code:', rc, '(0=pass)', flush=True)

for name in [
    'value_axis_32b.npy',
    'axis_manifest_32b.json',
    'auroc_by_layer_32b.json',
    'auroc_by_layer_32b.png',
]:
    src = data_file(name)
    if src.exists():
        dst = AXIS_DIR / name
        shutil.copy2(src, dst)
        print('copied to Drive:', dst, flush=True)
    else:
        print('missing locally:', src, flush=True)

In [ ]:
# Download artifacts (after gate)
# Faithful run: keep as value_axis_32b.npy for Stage 2.
# Proxy smoke: rename locally to e.g. value_axis_32b_smoke.npy so you do not overwrite a later faithful axis.

from google.colab import files
from stage1.common.paths import data_file
import json

manifest = data_file('axis_manifest_32b.json')
if manifest.exists():
    m = json.loads(manifest.read_text())
    print('gate_passed:', m.get('gate_passed'))
    print('primary_layer:', m.get('primary_layer'))
    print('icrl_path:', m.get('icrl_path'))
    print(json.dumps(m, indent=2)[:2000])
else:
    print('missing manifest — run gate first')

for name in [
    'value_axis_32b.npy',
    'axis_manifest_32b.json',
    'auroc_by_layer_32b.json',
    'auroc_by_layer_32b.png',
]:
    p = data_file(name)
    if p.exists():
        files.download(str(p))
    else:
        print('missing:', p)